In [222]:
# Imports et configuration
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
from tqdm import tqdm
import logging
import time
import warnings
import json
import os
import hashlib
import pickle
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from requests.exceptions import RequestException, Timeout
import optuna
from optuna.samplers import TPESampler
warnings.filterwarnings('ignore')

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('F1Predictor')

# Configuration du cache
CACHE_DIR = Path("f1_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Configuration des timeouts et retry
TIMEOUT = 10
RETRY_STRATEGY = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

In [223]:
# Fonctions de cache et API
def get_cached_session():
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=RETRY_STRATEGY)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def get_cache_path(year, round_num):
    cache_key = f"race_{year}_{round_num}"
    return CACHE_DIR / f"{cache_key}.pkl"

def save_to_cache(cache_path, data):
    try:
        with open(cache_path, 'wb') as f:
            pickle.dump(data, f)
        logger.debug(f"Données sauvegardées dans le cache: {cache_path}")
    except Exception as e:
        logger.warning(f"Erreur lors de la sauvegarde du cache: {str(e)}")

def load_from_cache(cache_path):
    try:
        if cache_path.exists():
            with open(cache_path, 'rb') as f:
                data = pickle.load(f)
            logger.debug(f"Données chargées depuis le cache: {cache_path}")
            return data
    except Exception as e:
        logger.warning(f"Erreur lors de la lecture du cache: {str(e)}")
    return None

def get_race_results(year, round_num):
    cache_path = get_cache_path(year, round_num)
    
    cached_data = load_from_cache(cache_path)
    if cached_data is not None:
        return cached_data
    
    url = f"http://ergast.com/api/f1/{year}/{round_num}/results.json"
    session = get_cached_session()
    
    try:
        response = session.get(url, timeout=TIMEOUT)
        response.raise_for_status()
        
        data = response.json()
        if 'MRData' in data and 'RaceTable' in data['MRData'] and 'Races' in data['MRData']['RaceTable']:
            race = data['MRData']['RaceTable']['Races']
            if race:
                save_to_cache(cache_path, race[0])
                return race[0]
                
    except Timeout:
        logger.error(f"Timeout lors de la récupération des données pour {year}-{round_num}")
    except RequestException as e:
        logger.error(f"Erreur réseau pour {year}-{round_num}: {str(e)}")
    except Exception as e:
        logger.error(f"Erreur inattendue pour {year}-{round_num}: {str(e)}")
    
    time.sleep(0.1)
    return None

In [224]:
# Collecte des données et feature engineering
def collect_historical_data(start_year=2010, end_year=2024, force_refresh=False):
    if force_refresh:
        logger.info("Force refresh activé - Le cache sera ignoré")
        for cache_file in CACHE_DIR.glob("race_*.pkl"):
            cache_file.unlink()
            
    logger.info(f"Début de la collecte des données de {start_year} à {end_year}")
    all_races = []
    total_years = end_year - start_year + 1
    
    with tqdm(total=total_years, desc="Années traitées") as year_pbar:
        for year in range(start_year, end_year + 1):
            races_this_year = 0
            logger.info(f"Traitement de l'année {year}")
            
            with tqdm(total=22, desc=f"Courses {year}", leave=False) as race_pbar:
                for round_num in range(1, 23):
                    race_data = get_race_results(year, round_num)
                    if race_data:
                        races_this_year += 1
                        race_info = {
                            'season': year,
                            'round': round_num,
                            'date': race_data['date'],
                            'circuitId': race_data['Circuit']['circuitId'],
                            'results': []
                        }
                        
                        for result in race_data['Results']:
                            driver_result = {
                                'driverId': result['Driver']['driverId'],
                                'constructorId': result['Constructor']['constructorId'],
                                'position': int(result['position']),
                                'status': result['status'],
                                'time': result.get('Time', {}).get('time', None),
                                'grid': int(result.get('grid', 0))
                            }
                            race_info['results'].append(driver_result)
                        
                        all_races.append(race_info)
                    race_pbar.update(1)
            
            logger.info(f"Année {year} terminée - {races_this_year} courses trouvées")
            year_pbar.update(1)
    
    logger.info(f"Collecte terminée - Total: {len(all_races)} courses")
    return all_races

def add_advanced_features(df):
    # Tendances
    for window in [5, 10]:
        start_idx = window - 1
        df[f'position_trend_{window}'] = df[f'position_{start_idx}'] - df['position_0']
    
    # Performances circuit/constructeur
    df['circuit_constructor_perf'] = df.groupby(['circuit', 'constructor'])['target_position'].transform('mean')
    
    # Stabilité des performances
    position_cols = [f'position_{i}' for i in range(5)]
    df['position_volatility'] = df[position_cols].std(axis=1)
    
    # DNF rate par circuit
    df['circuit_dnf_rate'] = df.groupby('circuit')['dnf_0'].transform('mean')
    
    return df

def create_driver_features(all_races, lookback=40):
    logger.info("Début de la création des features")
    features_list = []
    total_races = len(all_races)
    
    with tqdm(total=total_races, desc="Création des features") as pbar:
        for i, race in enumerate(all_races):
            historical_races = all_races[max(0, i-lookback):i]
            
            for current_result in race['results']:
                driver_id = current_result['driverId']
                features = {
                    'season': race['season'],
                    'round': race['round'],
                    'date': race['date'],
                    'driver': driver_id,
                    'constructor': current_result['constructorId'],
                    'circuit': race['circuitId'],
                    'grid': current_result['grid']
                }
                
                # Initialisation des historiques
                for pos_idx in range(lookback):
                    features[f'position_{pos_idx}'] = None
                    features[f'constructor_{pos_idx}'] = None
                    features[f'circuit_{pos_idx}'] = None
                    features[f'dnf_{pos_idx}'] = 0
                    features[f'time_gap_{pos_idx}'] = None
                
                # Remplissage des données historiques
                for hist_idx, hist_race in enumerate(historical_races):
                    for hist_result in hist_race['results']:
                        if hist_result['driverId'] == driver_id:
                            idx = len(historical_races) - hist_idx - 1
                            features[f'position_{idx}'] = hist_result['position']
                            features[f'constructor_{idx}'] = hist_result['constructorId']
                            features[f'circuit_{idx}'] = hist_race['circuitId']
                            features[f'dnf_{idx}'] = 1 if hist_result['status'] != 'Finished' else 0
                            if hist_result['time']:
                                try:
                                    time_gap = float(hist_result['time'].replace('+', '').replace(':', '.'))
                                    features[f'time_gap_{idx}'] = time_gap
                                except:
                                    features[f'time_gap_{idx}'] = None
                
                features['target_position'] = current_result['position']
                features_list.append(features)
            
            pbar.update(1)
    
    df = pd.DataFrame(features_list)
    df = add_advanced_features(df)
    logger.info(f"Features créées - Shape du dataset: {df.shape}")
    return df

In [225]:
# Préparation des données
def fill_missing_positions(df, col):
    # D'abord remplir avec la moyenne du pilote
    df[col] = df.groupby('driver')[col].transform(lambda x: x.fillna(x.mean()))
    # Si toujours des NA (nouveau pilote), mettre -1
    df[col] = df[col].fillna(-1)
    return df

def prepare_model_data(df):
    logger.info("Début de la préparation des données")
    df_model = df.copy()
    
    # Colonnes catégorielles pour LightGBM
    categorical_columns = ['driver', 'constructor', 'circuit'] + \
                         [f'constructor_{i}' for i in range(40)] + \
                         [f'circuit_{i}' for i in range(40)]
    
    # Conversion des colonnes catégorielles en type category de pandas
    logger.info("Conversion des colonnes catégorielles")
    for col in tqdm(categorical_columns, desc="Conversion catégories"):
        df_model[col] = df_model[col].astype('category')
    
    # Remplissage des valeurs manquantes
    logger.info("Remplissage des valeurs manquantes")
    position_columns = [f'position_{i}' for i in range(40)]
    time_gap_columns = [f'time_gap_{i}' for i in range(40)]
    
    with tqdm(total=len(position_columns) + len(time_gap_columns), 
             desc="Remplissage valeurs manquantes") as pbar:
        
        for col in position_columns:
            df_model = fill_missing_positions(df_model, col)
            pbar.update(1)
        
        for col in time_gap_columns:
            df_model[col] = df_model.groupby('driver')[col].transform(lambda x: x.fillna(x.mean()))
            df_model[col] = df_model[col].fillna(-1)
            pbar.update(1)
    
    logger.info(f"Préparation terminée - Shape final: {df_model.shape}")
    return df_model, categorical_columns

In [226]:
# Fonctions d'évaluation et d'ajustement
def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = spearmanr(y_true, y_pred)[0]
    return {'MAE': mae, 'RMSE': rmse, 'Spearman': spearman}

def adjust_predictions(predictions, race_data):
    df = pd.DataFrame({
        'driver': race_data['driver'],
        'predicted_raw': predictions,
        'last_position': race_data['position_0']
    })
    
    # Score composite
    df['ranking_score'] = (
        df['predicted_raw'] + 
        df['last_position'] * 0.005  # Très léger avantage au dernier résultat
    )
    return df['ranking_score'].rank(method='first')

def plot_performance_analysis(df_model, predictions, y_test, title=""):
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    
    # Performance par circuit
    sns.heatmap(df_model.pivot_table(
        index='driver', 
        columns='circuit', 
        values='target_position'
    ), ax=axes[0,0])
    axes[0,0].set_title('Performance par circuit et pilote')
    
    # Distribution des erreurs
    sns.histplot(predictions - y_test, ax=axes[0,1])
    axes[0,1].set_title('Distribution des erreurs de prédiction')
    
    # Prédiction vs Réalité
    sns.scatterplot(x=y_test, y=predictions, ax=axes[1,0])
    axes[1,0].set_title('Prédiction vs Réalité')
    
    # Performance temporelle
    temp_df = pd.DataFrame({'date': df_model['date'], 'error': predictions - y_test})
    sns.lineplot(data=temp_df.groupby('date')['error'].mean(), ax=axes[1,1])
    axes[1,1].set_title('Évolution temporelle de l\'erreur')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [227]:
# Modèles de prédiction
def train_grid_model(df_model, categorical_columns, params=None):
    """
    Entraîne un modèle pour prédire la position sur la grille
    params: paramètres optimisés par Optuna. Si None, utilise les paramètres par défaut
    """
    logger.info("Début de l'entraînement du modèle de grille")
    
    split_date = pd.to_datetime(df_model['date']).quantile(0.8)
    logger.info(f"Date de split pour grid: {split_date}")
    
    train_mask = pd.to_datetime(df_model['date']) <= split_date
    
    X_train = df_model[train_mask].drop(['target_position', 'season', 'round', 'date', 'grid'], axis=1)
    y_train = df_model[train_mask]['grid']
    X_test = df_model[~train_mask].drop(['target_position', 'season', 'round', 'date', 'grid'], axis=1)
    y_test = df_model[~train_mask]['grid']
    
    # Paramètres par défaut si pas de params optimisés
    if params is None:
        params = {
            'objective': 'poisson',
            'metric': 'mae',
            'num_leaves': 31,
            'learning_rate': 0.01,
            'feature_fraction': 0.8,
            'categorical_feature': categorical_columns
        }
    else:
        # S'assurer que les paramètres essentiels sont présents
        params['categorical_feature'] = categorical_columns
        params['objective'] = 'poisson'
        params['metric'] = 'mae'
    
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_columns)
    valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_columns, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=2000,
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(period=100)
        ]
    )
    
    raw_predictions = model.predict(X_test)
    adjusted_predictions = adjust_predictions(raw_predictions, df_model[~train_mask])
    metrics = evaluate_predictions(y_test, adjusted_predictions)
    
    logger.info(f"Entraînement terminé - Métriques: {metrics}")
    
    return model, adjusted_predictions, metrics, y_test

def train_position_model(df_model, categorical_columns, params=None):
    """
    Entraîne un modèle pour prédire la position finale
    params: paramètres optimisés par Optuna. Si None, utilise les paramètres par défaut
    """
    logger.info("Début de l'entraînement du modèle de position finale")
    
    split_date = pd.to_datetime(df_model['date']).quantile(0.8)
    train_mask = pd.to_datetime(df_model['date']) <= split_date
    
    X_train = df_model[train_mask].drop(['target_position', 'season', 'round', 'date'], axis=1)
    y_train = df_model[train_mask]['target_position']
    X_test = df_model[~train_mask].drop(['target_position', 'season', 'round', 'date'], axis=1)
    y_test = df_model[~train_mask]['target_position']
    
    if params is None:
        params = {
            'objective': 'poisson',
            'metric': 'mae',
            'num_leaves': 31,
            'learning_rate': 0.01,
            'feature_fraction': 0.8,
            'categorical_feature': categorical_columns
        }
    else:
        params['categorical_feature'] = categorical_columns
        params['objective'] = 'poisson'
        params['metric'] = 'mae'
    
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_columns)
    valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_columns, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=2000,
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(period=100)
        ]
    )
    
    raw_predictions = model.predict(X_test)
    adjusted_predictions = adjust_predictions(raw_predictions, df_model[~train_mask])
    metrics = evaluate_predictions(y_test, adjusted_predictions)
    
    logger.info(f"Entraînement terminé - Métriques: {metrics}")
    
    return model, adjusted_predictions, metrics, X_test, y_test

In [228]:
# Optimisation des hyperparamètres

def objective_grid(trial, df_model, categorical_columns):
    """Fonction objective pour l'optimisation du modèle de grille"""
    
    # Split temporel
    split_date = pd.to_datetime(df_model['date']).quantile(0.8)
    train_mask = pd.to_datetime(df_model['date']) <= split_date
    
    X_train = df_model[train_mask].drop(['target_position', 'season', 'round', 'date', 'grid'], axis=1)
    y_train = df_model[train_mask]['grid']
    X_val = df_model[~train_mask].drop(['target_position', 'season', 'round', 'date', 'grid'], axis=1)
    y_val = df_model[~train_mask]['grid']
    
    # Paramètres à optimiser
    params = {
        'objective': 'poisson',  # Distribution de Poisson pour les positions
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'max_bin': trial.suggest_int('max_bin', 200, 300),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'categorical_feature': categorical_columns
    }
    
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_columns)
    valid_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_columns, reference=train_data)
    
    # Entraînement
    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=20000,
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(period=100)
        ]
    )
    
    # Évaluation
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    
    return mae

def objective_position(trial, df_model, categorical_columns):
    """Fonction objective pour l'optimisation du modèle de position finale"""
    
    split_date = pd.to_datetime(df_model['date']).quantile(0.8)
    train_mask = pd.to_datetime(df_model['date']) <= split_date
    
    X_train = df_model[train_mask].drop(['target_position', 'season', 'round', 'date'], axis=1)
    y_train = df_model[train_mask]['target_position']
    X_val = df_model[~train_mask].drop(['target_position', 'season', 'round', 'date'], axis=1)
    y_val = df_model[~train_mask]['target_position']
    
    params = {
        'objective': 'poisson',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 1e-1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'max_bin': trial.suggest_int('max_bin', 200, 300),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 10.0),
        'categorical_feature': categorical_columns
    }
    
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_columns)
    valid_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_columns, reference=train_data)
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=20000,
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(period=100)
        ]
    )
    
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    
    return mae

def optimize_hyperparameters(df_model, categorical_columns, n_trials=100):
    """Optimise les hyperparamètres pour les deux modèles"""
    
    logger.info("Début de l'optimisation des hyperparamètres...")
    
    # Optimisation du modèle de grille
    logger.info("Optimisation du modèle de grille...")
    study_grid = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(),
        study_name="grid_position_optimization"
    )
    study_grid.optimize(
        lambda trial: objective_grid(trial, df_model, categorical_columns),
        n_trials=n_trials
    )
    
    # Optimisation du modèle de position
    logger.info("Optimisation du modèle de position finale...")
    study_position = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(),
        study_name="final_position_optimization"
    )
    study_position.optimize(
        lambda trial: objective_position(trial, df_model, categorical_columns),
        n_trials=n_trials
    )
    
    # Affichage des meilleurs paramètres
    logger.info("\nMeilleurs paramètres pour le modèle de grille:")
    print(study_grid.best_params)
    print(f"Meilleur score (MAE): {study_grid.best_value:.4f}")
    
    logger.info("\nMeilleurs paramètres pour le modèle de position finale:")
    print(study_position.best_params)
    print(f"Meilleur score (MAE): {study_position.best_value:.4f}")
    
    return study_grid.best_params, study_position.best_params

# Visualisation des résultats d'optimisation
def plot_optimization_history(study, title):
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.title(f'{title} - Historique d\'optimisation')
    
    plt.subplot(1, 2, 2)
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title(f'{title} - Importance des paramètres')
    
    plt.tight_layout()
    plt.show()

In [229]:
# Fonction de validation du cache
def validate_cache():
    """
    Valide l'intégrité du cache et supprime les fichiers corrompus
    """
    logger.info("Validation du cache...")
    corrupted_files = []
    
    for cache_file in CACHE_DIR.glob("race_*.pkl"):
        try:
            with open(cache_file, 'rb') as f:
                pickle.load(f)
        except Exception:
            corrupted_files.append(cache_file)
    
    if corrupted_files:
        logger.warning(f"Fichiers de cache corrompus trouvés: {len(corrupted_files)}")
        for file in corrupted_files:
            logger.warning(f"Suppression du fichier corrompu: {file}")
            file.unlink()
    else:
        logger.info("Cache validé - Aucun fichier corrompu trouvé")

In [ ]:
# Cellule 1 : Chargement des données
start_time = time.time()
logger.info("Début du processus")

# Validation du cache
validate_cache()

# Collecte des données
logger.info("Collecte des données historiques...")
all_races = collect_historical_data(force_refresh=False)
print(f"Nombre de courses collectées : {len(all_races)}")

In [ ]:
# Cellule 2 : Création et préparation des features
# Création des features
logger.info("Création des features...")
df = create_driver_features(all_races)
print(f"Shape initial du dataset : {df.shape}")

# Préparation des données
logger.info("Préparation des données...")
df_model, categorical_columns = prepare_model_data(df)
print(f"Shape final du dataset : {df_model.shape}")
print("\nNombre de colonnes catégorielles :", len(categorical_columns))

In [ ]:
# Cellule 3 : Optimisation des hyperparamètres
logger.info("Optimisation des hyperparamètres...")
best_params_grid, best_params_position = optimize_hyperparameters(df_model, categorical_columns, n_trials=50)

print("\nMeilleurs paramètres pour le modèle de grille:")
print(json.dumps(best_params_grid, indent=2))
print("\nMeilleurs paramètres pour le modèle de position:")
print(json.dumps(best_params_position, indent=2))

In [ ]:
# Cellule 4 : Entraînement du modèle de grille avec paramètres optimisés
logger.info("Entraînement du modèle de grille...")
grid_model, predicted_grid, grid_metrics, y_test_grid = train_grid_model(
    df_model, 
    categorical_columns,
    params=best_params_grid
)

print("\nMétriques du modèle de grille:")
for metric, value in grid_metrics.items():
    print(f"{metric}: {value:.3f}")

# Visualisation des performances du modèle de grille
plot_performance_analysis(df_model, predicted_grid, y_test_grid, "Analyse des prédictions de grille")

In [ ]:
# Cellule 5 : Ajout des prédictions de grille et entraînement du modèle de position
# Ajout des prédictions de grille comme feature
test_mask = pd.to_datetime(df_model['date']) > pd.to_datetime(df_model['date']).quantile(0.8)
df_model.loc[test_mask, 'predicted_grid'] = predicted_grid
df_model.loc[~test_mask, 'predicted_grid'] = df_model.loc[~test_mask, 'grid']

# Entraînement du modèle de position finale avec paramètres optimisés
logger.info("Entraînement du modèle de position finale...")
position_model, predictions, position_metrics, X_test, y_test = train_position_model(
    df_model, 
    categorical_columns,
    params=best_params_position
)

print("\nMétriques du modèle de position finale:")
for metric, value in position_metrics.items():
    print(f"{metric}: {value:.3f}")

# Visualisation des performances du modèle de position
plot_performance_analysis(df_model, predictions, y_test, "Analyse des prédictions de position finale")

In [ ]:
# Cellule 6 : Analyse de la dernière course
last_date = pd.to_datetime(df_model['date']).max()
last_race_mask = pd.to_datetime(df_model['date']) == last_date

results_df = pd.DataFrame({
    'Driver': df_model[last_race_mask]['driver'],
    'Constructor': df_model[last_race_mask]['constructor'],
    'Circuit': df_model[last_race_mask]['circuit'],
    'Actual Grid': df_model[last_race_mask]['grid'],
    'Predicted Grid': df_model[last_race_mask]['predicted_grid'],
    'Actual Position': df_model[last_race_mask]['target_position'],
    'Predicted Position': predictions[-len(df_model[last_race_mask]):]
}).sort_values('Predicted Position')

print("\nDernière course - Résultats complets:")
print(results_df)

In [ ]:
# Cellule 7 : Analyse des features importantes et résultats finaux
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance': position_model.feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 features les plus importantes:")
print(feature_importance.head(10))

# Visualisation des features importantes
plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Features les plus importantes')
plt.tight_layout()
plt.show()

# Temps d'exécution total
end_time = time.time()
logger.info(f"\nTemps total d'exécution: {(end_time - start_time)/60:.2f} minutes")

In [ ]:
# Nouvelle fonction pour obtenir les prédictions futures
def predict_future_races(df_model, grid_model, position_model, categorical_columns, start_year=2024, start_round=16):
    """
    Prédit les résultats pour toutes les courses à partir d'une date donnée
    """
    logger.info(f"Prédiction des courses à partir de {start_year} round {start_round}")
    
    # Identification des courses futures
    future_mask = ((df_model['season'] == start_year) & (df_model['round'] >= start_round))
    
    # DataFrame pour stocker les résultats
    all_predictions = []
    
    # Grouper par course
    for (year, round_num), race_df in df_model[future_mask].groupby(['season', 'round']):
        logger.info(f"Prédiction course {year} - Round {round_num}")
        
        # Prédiction de la grille
        X_race = race_df.drop(['target_position', 'season', 'round', 'date', 'grid'], axis=1)
        grid_preds_raw = grid_model.predict(X_race)
        grid_preds = adjust_predictions(grid_preds_raw, race_df)
        
        # Ajout des prédictions de grille
        race_df = race_df.copy()
        race_df['predicted_grid'] = grid_preds
        
        # Prédiction de la position finale
        X_race_final = race_df.drop(['target_position', 'season', 'round', 'date'], axis=1)
        position_preds_raw = position_model.predict(X_race_final)
        position_preds = adjust_predictions(position_preds_raw, race_df)
        
        # Création du DataFrame de résultats
        results = pd.DataFrame({
            'Season': year,
            'Round': round_num,
            'Circuit': race_df['circuit'],
            'Driver': race_df['driver'],
            'Constructor': race_df['constructor'],
            'Predicted_Grid': grid_preds,
            'Actual_Grid': race_df['grid'],
            'Predicted_Position': position_preds,
            'Actual_Position': race_df['target_position']
        })
        
        all_predictions.append(results)
    
    return pd.concat(all_predictions, ignore_index=True)

In [ ]:
# Cellule pour les prédictions futures
future_predictions = predict_future_races(
    df_model, 
    grid_model, 
    position_model, 
    categorical_columns,
    start_year=2024,
    start_round=16
)

# Affichage des résultats par course
for (year, round_num), race_results in future_predictions.groupby(['Season', 'Round']):
    print(f"\nPrédictions pour {year} - Round {round_num}")
    print("Circuit:", race_results['Circuit'].iloc[0])
    print("\nGrille de départ prédite:")
    grid_results = race_results.sort_values('Predicted_Grid')
    print(grid_results[['Driver', 'Constructor', 'Predicted_Grid']].to_string(index=False))
    
    print("\nPosition finale prédite:")
    position_results = race_results.sort_values('Predicted_Position')
    print(position_results[['Driver', 'Constructor', 'Predicted_Position']].to_string(index=False))
    
    if not race_results['Actual_Position'].isna().all():
        print("\nComparaison avec les résultats réels:")
        comparison = race_results[['Driver', 'Predicted_Position', 'Actual_Position']].sort_values('Actual_Position')
        print(comparison.to_string(index=False))
    print("\n" + "="*50)

# Analyse des performances prédictives si des résultats réels sont disponibles
real_results_mask = ~future_predictions['Actual_Position'].isna()
if real_results_mask.any():
    real_results = future_predictions[real_results_mask]
    metrics = evaluate_predictions(
        real_results['Actual_Position'], 
        real_results['Predicted_Position']
    )
    print("\nMétriques sur les courses réalisées:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.3f}")